In [2]:
# ---------------------------------------
# 1. Imports
# ---------------------------------------
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm
import itertools

# ---------------------------------------
# 2. Load Data
# ---------------------------------------
train_data_encoded = pd.read_csv('../data/train_encoded.csv')
X = train_data_encoded.drop('y', axis=1).values
y = train_data_encoded['y'].values

# Log-transform target to handle skew
y_log = np.log1p(y)

# ---------------------------------------
# 3. Parameter Grid
# ---------------------------------------
param_grid = {
    'C': [0.1, 1, 10],
    'epsilon': [0.01, 0.1, 1],
    'gamma': ['scale', 'auto']
}

param_list = list(itertools.product(param_grid['C'], param_grid['epsilon'], param_grid['gamma']))

# ---------------------------------------
# 4. Cross-Validation Setup
# ---------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_mse = np.inf
best_params = None
best_model = None

# ---------------------------------------
# 5. Manual Grid Search with Progress Bar
# ---------------------------------------
for C_val, epsilon_val, gamma_val in tqdm(param_list, desc="SVR Hyperparam tuning"):
    fold_mse = []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y_log[train_idx], y_log[val_idx]

        # Pipeline: scaling + SVR
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('svr', SVR(kernel='rbf', C=C_val, epsilon=epsilon_val, gamma=gamma_val))
        ])

        pipeline.fit(X_tr, y_tr)

        y_val_pred_log = pipeline.predict(X_val)
        mse_fold = mean_squared_error(y_val, y_val_pred_log)
        fold_mse.append(mse_fold)

    avg_mse = np.mean(fold_mse)

    if avg_mse < best_mse:
        best_mse = avg_mse
        best_params = (C_val, epsilon_val, gamma_val)
        best_model = pipeline

# ---------------------------------------
# 6. Results
# ---------------------------------------
print("\nBest SVR hyperparameters:")
print(f"C: {best_params[0]}, epsilon: {best_params[1]}, gamma: {best_params[2]}")
print(f"CV MSE: {best_mse:.4f}")

# ---------------------------------------
# 7. Evaluate on Full Training Set
# ---------------------------------------
y_log_pred_train = best_model.predict(X)
y_pred_train = np.expm1(y_log_pred_train)

mse = mean_squared_error(y, y_pred_train)
r2 = r2_score(y, y_pred_train)

print("\nSVR Full Training Performance:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.4f}")

# ---------------------------------------
# 8. Predict on Test Set
# ---------------------------------------
X_test = pd.read_csv('../data/test_encoded.csv').values

y_log_test_pred = best_model.predict(X_test)
y_test_pred = np.expm1(y_log_test_pred)

# Clip negative predictions
y_test_pred = np.clip(y_test_pred, 0, None)

# ---------------------------------------
# 9. Save Predictions
# ---------------------------------------
os.makedirs('../prediction', exist_ok=True)
np.savetxt('../prediction/predicted_SVR.txt', y_test_pred, fmt='%.6f')

print("\nFinal SVR predictions saved to '../prediction/predicted_SVR.txt'.")


SVR Hyperparam tuning: 100%|██████████| 18/18 [13:08<00:00, 43.79s/it]



Best SVR hyperparameters:
C: 1, epsilon: 0.1, gamma: scale
CV MSE: 1.5049

SVR Full Training Performance:
Mean Squared Error (MSE): 4972.51
R² Score: 0.7621

Final SVR predictions saved to '../prediction/predicted_SVR.txt'.
